In [1]:
RANDOM_SEED = 414

import warnings
from typing import List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import fastf1

warnings.filterwarnings("ignore")
np.random.seed(RANDOM_SEED)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

In [2]:
YEARS = [2022, 2023, 2024]

rows = []

for year in YEARS:
    schedule = fastf1.get_event_schedule(year, include_testing=False)

    for round_number in schedule["RoundNumber"].dropna().astype(int).unique():
        try:
            session = fastf1.get_session(year, round_number, "R")
            session.load(laps=False, telemetry=False, weather=False, messages=False)

            race = session.results[[
                "Abbreviation",
                "FullName",
                "TeamName",
                "GridPosition",
                "Position"
            ]].copy()

            race["season"] = year
            race["round"] = round_number
            rows.append(race)

        except Exception as e:
            print(f"Could not load {year} round {round_number}: {e}")

df = pd.concat(rows, ignore_index=True)

df = df.rename(columns={
    "Abbreviation": "driver_code",
    "FullName": "driver_name",
    "TeamName": "team_name",
    "GridPosition": "grid_position",
    "Position": "finish_position"
})

df["grid_position"] = pd.to_numeric(df["grid_position"], errors="coerce")
df["finish_position"] = pd.to_numeric(df["finish_position"], errors="coerce")

df = df[df["finish_position"].notna()].copy()
df["is_top10"] = (df["finish_position"] <= 10).astype(int)

df.head()

req         WARNING 	DEFAULT CACHE ENABLED! (2.78 MB) C:\Users\dinot\AppData\Local\Temp\fastf1
core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['16', '55', '44', '63', '20', '77', '31', '22', '14', '24', '47', '18', '23', '3', '4', '6', '27', '11', '1', '10']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '55', '11', '63', '31', '4', '10', '20', '44', '24', '27', '18', '23', '77', '14', '3', '6', '22', '47']
core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driv

,driver_code,driver_name,team_name,grid_position,finish_position,season,round,is_top10
0,LEC,Charles Leclerc,Ferrari,1.0,1.0,2022,1,1
1,SAI,Carlos Sainz,Ferrari,3.0,2.0,2022,1,1
2,HAM,Lewis Hamilton,Mercedes,5.0,3.0,2022,1,1
3,RUS,George Russell,Mercedes,9.0,4.0,2022,1,1
4,MAG,Kevin Magnussen,Haas F1 Team,7.0,5.0,2022,1,1


## 4.1 Domain Heuristic Baseline

Rule:

If a driver starts in the Top 10 (grid ≤ 10), predict that they will finish in the Top 10.
Otherwise, predict that they will not finish in the Top 10.

This rule is based on domain knowledge: drivers starting near the front usually finish near the front.

In [3]:
validation_df = df[df["season"] == 2023].copy()

validation_df["prediction"] = (validation_df["grid_position"] <= 10).astype(int)

correct_predictions = (validation_df["prediction"] == validation_df["is_top10"]).sum()
total_predictions = len(validation_df)
baseline_accuracy = correct_predictions / total_predictions

always_top10_accuracy = (validation_df["is_top10"] == 1).mean()
always_non_top10_accuracy = (validation_df["is_top10"] == 0).mean()

print("Validation season:", validation_df["season"].iloc[0])
print(f"Correct predictions: {correct_predictions}/{total_predictions}")
print(f"Baseline accuracy (grid <= 10 rule): {baseline_accuracy:.3f}")
print(f"Naive always predict Top-10 accuracy: {always_top10_accuracy:.3f}")
print(f"Naive always predict Non-Top-10 accuracy: {always_non_top10_accuracy:.3f}")

validation_df[["season", "round", "driver_code", "grid_position", "is_top10", "prediction"]].head()

Validation season: 2023
Correct predictions: 321/439
Baseline accuracy (grid <= 10 rule): 0.731
Naive always predict Top-10 accuracy: 0.501
Naive always predict Non-Top-10 accuracy: 0.499


,season,round,driver_code,grid_position,is_top10,prediction
440,2023,1,VER,1.0,1,1
441,2023,1,PER,2.0,1,1
442,2023,1,ALO,5.0,1,1
443,2023,1,SAI,4.0,1,1
444,2023,1,HAM,7.0,1,1


## 4.2 Accuracy on Validation Set

Accuracy is computed on the validation set only (season 2023) using:

accuracy = correct predictions / total predictions


The code cell above reports this value explicitly as:
**Baseline accuracy (grid <= 10 rule): ...**

## 4.3 Reflection on Accuracy

Is this accuracy good enough for decision making? It is a useful minimum reference, but not enough by itself.
Accuracy can hide important behavior, especially if classes are close to 50/50 in F1 Top 10 outcomes.

If around 50% of drivers finish Top 10, a naive model that always predicts Top 10 would score around 0.50 accuracy.
That is why we compare the heuristic baseline to naive references and interpret accuracy carefully, not in isolation.

## 4.4 Baseline as Lower Bound

Any model we build in Lab 2 must beat this baseline accuracy number. If it does not, the model adds no value.

## 4.5 No Leakage (Pre race Features Only)

This baseline uses only `grid_position` to generate predictions, which is known before race start.

Excluded from prediction logic (post-race / leakage): `finish_position`, points, and any race outcome fields.

Therefore, the baseline is leakage safe by construction for a pre race decision setting.

## 4.6 Additional Metrics (Burkov Ch. 2)

To evaluate the heuristic baseline beyond accuracy, we compute:

- **Precision**: among predicted Top-10, how many were actually Top-10.
- **Recall**: among actual Top-10, how many were correctly predicted.
- **F1-score**: harmonic mean of Precision and Recall.
- **ROC-AUC**: ranking quality metric across thresholds (using a pre-race score).

These metrics are computed on the **validation set (2023)** only.

In [4]:
y_true = validation_df["is_top10"].astype(int)
y_pred = validation_df["prediction"].astype(int)

tp = int(((y_true == 1) & (y_pred == 1)).sum())
fp = int(((y_true == 0) & (y_pred == 1)).sum())
tn = int(((y_true == 0) & (y_pred == 0)).sum())
fn = int(((y_true == 1) & (y_pred == 0)).sum())

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

# Pre-race continuous score for ROC-AUC: better grid position -> higher score
scores = -validation_df["grid_position"].fillna(validation_df["grid_position"].max() + 1)

# ROC-AUC via rank statistic (equivalent to Mann-Whitney U formulation)
n_pos = int((y_true == 1).sum())
n_neg = int((y_true == 0).sum())

if n_pos > 0 and n_neg > 0:
    ranks = scores.rank(method="average")
    sum_ranks_pos = ranks[y_true == 1].sum()
    roc_auc = (sum_ranks_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)
else:
    roc_auc = np.nan

print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-score: {f1:.3f}")
print(f"ROC-AUC: {roc_auc:.3f}" if not np.isnan(roc_auc) else "ROC-AUC: undefined (single class)")
print(f"Confusion matrix (TN, FP, FN, TP): ({tn}, {fp}, {fn}, {tp})")

Precision: 0.730
Recall: 0.736
F1-score: 0.733
ROC-AUC: 0.797
Confusion matrix (TN, FP, FN, TP): (159, 60, 58, 162)


### Interpretation for Decision-Making

- Accuracy can look acceptable even when the model misses many true Top-10 outcomes.
- Precision and Recall show different operational trade-offs.
- F1-score summarizes balance between false positives and false negatives.
- ROC-AUC helps evaluate ranking power independently of a single cutoff.

For Lab 2, we should compare candidate models against this full metric set, not only accuracy.